PreProcessing for Task A

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

In [ ]:
# medical_charges_url = "https://raw.githubusercontent.com/JovianML/opendatasets/master/data/medical-charges.csv"
# urlretrieve(medical_charges_url, "medical.csv")

medical_df = pd.read_csv("medical.csv")
medical_df.head()

(EDA)

In [ ]:
# Statistical summary
print("Statistical Summary:")
print(medical_df.describe(include="all"))
medical_df.info()

In [ ]:
# Missing values count
print("Missing values count:")
print(medical_df.isnull().sum())

In [ ]:
# Duplicate rows count and removal
print("Number of duplicate rows:", medical_df.duplicated().sum())
medical_df.drop_duplicates(inplace=True)
print("Number of duplicate rows:", medical_df.duplicated().sum())

In [ ]:
# Value counts
print("Value counts for 'sex':")
print(medical_df["sex"].value_counts())

print("Value counts for 'smoker':")
print(medical_df["smoker"].value_counts())

print("Value counts for 'region':")
print(medical_df["region"].value_counts())

In [ ]:
# Average charges by smoker
print("Average charges by smoker:")
print(medical_df.groupby("smoker")["charges"].mean())

# Average charges by sex and smoker
print("Average charges by sex and smoker:")
print(medical_df.groupby(["sex", "smoker"])["charges"].mean())

In [ ]:
# Correlation with charges
print("Correlation with charges:")
print(
    medical_df.corr(numeric_only=True)["charges"]
    .sort_values(ascending=False)
)

Outlier Detection

In [ ]:
Q1 = medical_df["charges"].quantile(0.25)
Q3 = medical_df["charges"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = medical_df[
    (medical_df["charges"] < lower) |
    (medical_df["charges"] > upper)
]

print("Number of outliers:", len(outliers))
print(
    "Outliers percentage:",
    int(len(outliers) / len(medical_df) * 100),
    "%"
)

Plotting 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Charges outliers
sns.boxplot(x=medical_df["charges"], ax=axes[0])
axes[0].set_title("Charges Outlier Detection")
axes[0].set_xlabel("Charges")

# 2. Relationship between Age and Charges
axes[1].scatter(
    medical_df["age"],
    medical_df["charges"],
    alpha=0.5
)
axes[1].set_title("Age vs Charges")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Charges")

# 3. Relationship between BMI and Charges
axes[2].scatter(
    medical_df["bmi"],
    medical_df["charges"],
    alpha=0.5
)
axes[2].set_title("BMI vs Charges")
axes[2].set_xlabel("BMI")
axes[2].set_ylabel("Charges")

plt.tight_layout()
plt.show()

Data Cleaning

In [ ]:
# Remove rows with missing target values
medical_df.dropna(subset=["charges"], inplace=True)

# Separate smokers and non-smokers
non_smoker = medical_df[medical_df["smoker"] == "no"]
smoker = medical_df[medical_df["smoker"] == "yes"]

In [ ]:
medical_df_without_charges = medical_df.drop(columns=["charges"])
inputs = medical_df_without_charges
target = medical_df["charges"]

num_cols = inputs.select_dtypes(include=["number"]).columns.tolist()
cat_cols = inputs.select_dtypes(include=["object", "category"]).columns.tolist()

# Optional: inspect selected columns
# print("Numeric columns:", num_cols)
# print("Categorical columns:", cat_cols)

Split Dataset into Train, Validation, and Test Sets (70/15/15)

In [ ]:
# Split before preprocessing to avoid data leakage
X_train, X_temp, y_train, y_temp = train_test_split(
    inputs,
    target,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

 Preprocessing Pipelines

In [ ]:
# Numerical pipeline
num_pipes = make_pipeline(
    SimpleImputer(strategy="mean"),
    MinMaxScaler()
)

# Categorical pipeline
cat_pipes = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        drop="if_binary"
    )
)

# Apply the appropriate pipeline to each column type
preprocessing = ColumnTransformer([
    ("num", num_pipes, num_cols),
    ("cat", cat_pipes, cat_cols)
])

In [ ]:
train_pre = preprocessing.fit_transform(X_train)
val_pre = preprocessing.transform(X_val)
test_pre = preprocessing.transform(X_test)

In [ ]:
enc_cols = list(
    preprocessing.named_transformers_["cat"]
    .named_steps["onehotencoder"]
    .get_feature_names_out(cat_cols)
)

In [ ]:
X_train_pre = pd.DataFrame(
    train_pre,
    columns=num_cols + enc_cols,
    index=X_train.index
)

X_val_pre = pd.DataFrame(
    val_pre,
    columns=num_cols + enc_cols,
    index=X_val.index
)

X_test_pre = pd.DataFrame(
    test_pre,
    columns=num_cols + enc_cols,
    index=X_test.index
)

print("Training set shape:", X_train_pre.shape)
print("Validation set shape:", X_val_pre.shape)
print("Test set shape:", X_test_pre.shape)